# Fund Analysis
This notebook analyzes fund performance relative to the SP500.

In [ ]:
# SET THE FUND COLUMN NAME HERE
FUND_NAME = 'gator'

In [ ]:
import pandas as pd
import numpy as np
import statsmodels.api as sm
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid")

# Load data
df = pd.read_csv('../data/concat_hf_data.csv')
if 'date' in df.columns:
    df['date'] = pd.to_datetime(df['date'])
    df = df.set_index('date')

# Keep only sp500 and target fund columns
fund_col = FUND_NAME
df = df[['sp500', fund_col]].dropna()

sp500 = df['sp500']
fund = df[fund_col]

df.head()

In [ ]:
# Trading Metrics
mean_ret = fund.mean() * 12
vol = fund.std() * np.sqrt(12)
sharpe = mean_ret / vol if vol != 0 else np.nan

# CAGR
cum_ret = (1 + fund).prod()
n_years = len(fund) / 12
cagr = cum_ret**(1/n_years) - 1

# Max Drawdown
cum_wealth = (1 + fund).cumprod()
peak = cum_wealth.cummax()
drawdown = (cum_wealth - peak) / peak
max_drawdown = drawdown.min()

# Calmar Ratio
calmar = cagr / abs(max_drawdown) if max_drawdown != 0 else np.nan

print("--- Trading Metrics ---")
print(f"Annualized Mean Return: {mean_ret:.4f}")
print(f"Annualized Volatility:  {vol:.4f}")
print(f"Sharpe Ratio:           {sharpe:.4f}")
print(f"CAGR:                   {cagr:.4f}")
print(f"Max Drawdown:           {max_drawdown:.4f}")
print(f"Calmar Ratio:           {calmar:.4f}")

In [ ]:
# Joint Metrics
pearson_corr = fund.corr(sp500, method='pearson')
spearman_corr = fund.corr(sp500, method='spearman')

X = sm.add_constant(sp500)
model = sm.OLS(fund, X).fit()
alpha = model.params['const'] * 12
beta = model.params['sp500']
alpha_tstat = model.tvalues['const']
beta_tstat = model.tvalues['sp500']

print("--- Joint Metrics ---")
print(f"Pearson Correlation:  {pearson_corr:.4f}")
print(f"Spearman Correlation: {spearman_corr:.4f}")
print(f"Regression Alpha (Annualized): {alpha:.4f} (t-stat: {alpha_tstat:.4f})")
print(f"Regression Beta:               {beta:.4f} (t-stat: {beta_tstat:.4f})")

In [ ]:
# Upside / Downside Capture
up_months = df[df['sp500'] > 0]
down_months = df[df['sp500'] < 0]

up_capture = up_months[fund_col].mean() / up_months['sp500'].mean()
down_capture = down_months[fund_col].mean() / down_months['sp500'].mean()

print("--- Capture Ratios ---")
print(f"Upside Capture:   {up_capture:.4f}")
print(f"Downside Capture: {down_capture:.4f}")

In [ ]:
# Conditional Metrics: Positive / Negative SP500 Months
model_up = sm.OLS(up_months[fund_col], sm.add_constant(up_months['sp500'])).fit()
print("--- Up Months Metrics ---")
print(f"Up Alpha (Annualized): {model_up.params['const'] * 12:.4f} (t-stat: {model_up.tvalues['const']:.4f})")
print(f"Up Beta:               {model_up.params['sp500']:.4f} (t-stat: {model_up.tvalues['sp500']:.4f})")

print("\n--- Down Months Metrics ---")
model_down = sm.OLS(down_months[fund_col], sm.add_constant(down_months['sp500'])).fit()
print(f"Down Alpha (Annualized): {model_down.params['const'] * 12:.4f} (t-stat: {model_down.tvalues['const']:.4f})")
print(f"Down Beta:               {model_down.params['sp500']:.4f} (t-stat: {model_down.tvalues['sp500']:.4f})")

In [ ]:
# Correlation when SP500 is lower than 1st decile and 1st quartile
decile_1 = sp500.quantile(0.10)
df_d1 = df[df['sp500'] < decile_1]
d1_pearson = df_d1[fund_col].corr(df_d1['sp500'], method='pearson')
d1_spearman = df_d1[fund_col].corr(df_d1['sp500'], method='spearman')

quartile_1 = sp500.quantile(0.25)
df_q1 = df[df['sp500'] < quartile_1]
q1_pearson = df_q1[fund_col].corr(df_q1['sp500'], method='pearson')
q1_spearman = df_q1[fund_col].corr(df_q1['sp500'], method='spearman')

print("--- Tail Correlations ---")
print(f"SP500 < 1st Decile   | Pearson: {d1_pearson:.4f}, Spearman: {d1_spearman:.4f}")
print(f"SP500 < 1st Quartile | Pearson: {q1_pearson:.4f}, Spearman: {q1_spearman:.4f}")

In [ ]:
# Scatter plot
plt.figure(figsize=(10, 6))
plt.scatter(sp500, fund, alpha=0.7, color='steelblue')
plt.axvline(0, color='grey', linestyle='--', linewidth=1)
plt.axhline(0, color='grey', linestyle='--', linewidth=1)
plt.xlabel('SP500 Returns')
plt.ylabel(f'{fund_col} Returns')
plt.title(f'Scatter plot of {fund_col} Returns vs SP500 Returns')

# Regression line
x_vals = np.linspace(sp500.min(), sp500.max(), 100)
y_vals = model.params['sp500'] * x_vals + model.params['const']
plt.plot(x_vals, y_vals, color='firebrick', linewidth=2, label=f"Fit: y = {model.params['sp500']:.2f}x + {model.params['const'] * 12:.4f} (Ann. Alpha)")

plt.legend()
plt.tight_layout()
plt.show()